In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:


!mkdir -p ~/.kaggle                                           # Create .kaggle folder if it doesn't exist
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json >log  # Copy kaggle.json from Google Drive
!cp kaggle.json ~/.kaggle/kaggle.json > log                # Uncomment if using kaggle.json directly from local path
!chmod 600 ~/.kaggle/kaggle.json                              # Secure kaggle.json
!kaggle config set -n competition -v cmi-detect-behavior-with-sensor-data  # Set the active competition
!kaggle competitions download -c cmi-detect-behavior-with-sensor-data >> log  # Download competition dataset
!unzip -o *.zip >> log                                        # Unzip the dataset
!kaggle competitions leaderboard -c cmi-detect-behavior-with-sensor-data --show  # Show the leaderboard

cp: cannot stat 'kaggle.json': No such file or directory
- competition is now set to: cmi-detect-behavior-with-sensor-data
100% 178M/178M [00:00<00:00, 947MB/s] 
  teamId  teamName          submissionDate              score  
--------  ----------------  --------------------------  -----  
14060040  Psi               2025-08-11 19:10:28.103000  0.877  
13956663  Anil + Optimo     2025-08-11 13:16:10.316000  0.876  
13896136  ln | minerppdy    2025-08-11 17:02:11.350000  0.875  
13897295  Devin Anzelmo     2025-08-10 13:00:55.903000  0.874  
13905890  lightsource<3     2025-08-11 18:25:02.046000  0.872  
13930188  Ogurtsov          2025-08-11 18:02:40.216000  0.870  
13907397  D.Imanishi        2025-08-11 15:45:11.326000  0.870  
13944810  ryches            2025-08-10 23:26:30.793000  0.870  
13895851  根本打不过             2025-08-11 11:54:32.610000  0.868  
13934925  Wrist Wizards!    2025-08-11 12:15:19.593000  0.868  
13895645  Takoi             2025-08-11 11:28:11.046000  0.868  
139078

In [ ]:
import numpy as np
import pandas as pd
import os
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

In [ ]:
train_data_df = pd.read_csv("train.csv")
train_demo_df = pd.read_csv("train_demographics.csv")

In [ ]:
test_data_df = pd.read_csv("test.csv")
test_demo_df = pd.read_csv("test_demographics.csv")

In [ ]:
file_size_bytes = os.path.getsize("train.csv")
file_size_mb = file_size_bytes / (1024 * 1024*1024)
print(f"File size: {file_size_mb:.2f} MB")

In [ ]:
len(test_data_df.sequence_id.unique())

2

In [ ]:
len(test_demo_df.columns)


8

In [ ]:
SEED = 42
np.random.seed(SEED)

In [ ]:
test_data_df = pd.read_csv("test.csv")
test_demo_df = pd.read_csv("test_demographics.csv")

In [ ]:
test_data_df["sequence_id"].unique()

array(['SEQ_000001', 'SEQ_000011'], dtype=object)

In [ ]:
cols = [
    "row_id",
    "sequence_id",
    "sequence_type",
    "sequence_counter",
    "subject",
    "gesture",
    "orientation",
    "phase"
]

# Show first 5 rows
train_data_df[cols].head()

,row_id,sequence_id,sequence_type,sequence_counter,subject,gesture,orientation,phase
0,SEQ_000007_000000,SEQ_000007,Target,0,SUBJ_059520,Cheek - pinch skin,Seated Lean Non Dom - FACE DOWN,Transition
1,SEQ_000007_000001,SEQ_000007,Target,1,SUBJ_059520,Cheek - pinch skin,Seated Lean Non Dom - FACE DOWN,Transition
2,SEQ_000007_000002,SEQ_000007,Target,2,SUBJ_059520,Cheek - pinch skin,Seated Lean Non Dom - FACE DOWN,Transition
3,SEQ_000007_000003,SEQ_000007,Target,3,SUBJ_059520,Cheek - pinch skin,Seated Lean Non Dom - FACE DOWN,Transition
4,SEQ_000007_000004,SEQ_000007,Target,4,SUBJ_059520,Cheek - pinch skin,Seated Lean Non Dom - FACE DOWN,Transition


In [ ]:
train_demo_df.head()

,subject,adult_child,age,sex,handedness,height_cm,shoulder_to_wrist_cm,elbow_to_wrist_cm
0,SUBJ_000206,1,41,1,1,172.0,50,25.0
1,SUBJ_001430,0,11,0,1,167.0,51,27.0
2,SUBJ_002923,1,28,1,0,164.0,54,26.0
3,SUBJ_003328,1,33,1,1,171.0,52,25.0
4,SUBJ_004117,0,15,0,1,184.0,54,28.0


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Select first 5 timesteps
df_first5 = train_data_df.iloc[:5]

# Number of sensors and pixels per sensor
num_sensors = 5
pixels_per_sensor = 64  # 8x8

gesture_label = df_first5['gesture'].iloc[0]

for timestep_idx, (_, row) in enumerate(df_first5.iterrows()):
    fig = make_subplots(
        rows=1, cols=num_sensors,
        subplot_titles=[f"ToF Sensor {i+1}" for i in range(num_sensors)],
        horizontal_spacing=0.05
    )

    for sensor_idx in range(num_sensors):
        # Extract 64 values for this sensor at this timestep
        pixel_cols = [f"tof_{sensor_idx+1}_v{v}" for v in range(pixels_per_sensor)]
        pixel_values = row[pixel_cols].values

        # Replace -1 with np.nan for plotting (no response)
        pixel_values = np.where(pixel_values == -1, np.nan, pixel_values)

        # Reshape into 8x8 grid
        grid = pixel_values.reshape((8,8))

        # Add heatmap to subplot
        fig.add_trace(
            go.Heatmap(
                z=grid,
                zmin=0, zmax=254,
                colorscale='Viridis',
                colorbar=dict(title='Proximity'),
                showscale=(sensor_idx==num_sensors-1),  # show colorbar only for last sensor
                hoverongaps=False,
            ),
            row=1, col=sensor_idx+1
        )

    fig.update_layout(
        title_text=f"ToF Sensor Readings at Timestep {timestep_idx} (sequence_id {row['sequence_id']}) — Gesture: {gesture_label}",
        height=350,
        width=1200,
        template='plotly_dark',
    )

    fig.show()


In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Filter one sequence
sequence_id = train_data_df['sequence_id'].iloc[0]
df_seq = train_data_df[train_data_df['sequence_id'] == sequence_id].copy()

# Melt the sensor columns
sensor_cols = ['rot_x', 'rot_y', 'rot_z', 'rot_w', 'acc_x', 'acc_y', 'acc_z']
df_melted = df_seq.melt(
    id_vars=['sequence_counter', 'phase'],
    value_vars=sensor_cols,
    var_name='sensor',
    value_name='value'
)

# Get gesture label for the current sequence (assumes constant within sequence)
gesture_label = df_seq['gesture'].iloc[0]

# Define color palettes
acc_colors = ['#0d47a1', '#1976d2', '#42a5f5']  # dark to lighter blue for acc_x, acc_y, acc_z
rot_colors = ['#ef6c00', '#ff9800', '#ffb74d', '#ffe0b2']  # dark to lighter orange for rot_x, rot_y, rot_z, rot_w

# Map sensor names to colors
color_map = {
    'acc_x': acc_colors[0],
    'acc_y': acc_colors[1],
    'acc_z': acc_colors[2],
    'rot_x': rot_colors[0],
    'rot_y': rot_colors[1],
    'rot_z': rot_colors[2],
    'rot_w': rot_colors[3],
}

# Create base line plot with custom colors and dynamic title
fig = px.line(
    df_melted,
    x='sequence_counter',
    y='value',
    color='sensor',
    color_discrete_map=color_map,
    title=f"Sensor Readings for Sequence ID {sequence_id} — Gesture: {gesture_label}",
    labels={'sequence_counter': 'Time Step', 'value': 'Sensor Value'},
    template='plotly_dark',
    width=900,
    height=500
)

# Colors for phases
phase_colors = {
    'Transition': 'rgba(255,165,0,0.15)',  # orange
    'Pause': 'rgba(0,0,255,0.15)',         # blue
    'Gesture': 'rgba(0,255,0,0.15)'        # green
}

min_width = 5  # Minimum width in timesteps for label placement

# Add shaded phase zones and labels
for phase in df_seq['phase'].unique():
    phase_rows = df_seq[df_seq['phase'] == phase]

    block_start = None
    prev_counter = None

    for _, row in phase_rows.iterrows():
        if block_start is None:
            block_start = row['sequence_counter']
            prev_counter = row['sequence_counter']
        elif row['sequence_counter'] != prev_counter + 1:
            # Add shaded zone
            fig.add_vrect(
                x0=block_start, x1=prev_counter,
                fillcolor=phase_colors[phase],
                opacity=0.3,
                layer="below",
                line_width=0
            )
            # Calculate label x-position with minimum width adjustment
            zone_width = prev_counter - block_start + 1
            mid_point = (block_start + prev_counter) / 2
            if zone_width < min_width:
                label_x = block_start + min_width / 2
            else:
                label_x = mid_point
            # Add label annotation
            fig.add_annotation(
                x=label_x,
                y=1.05,
                text=phase,
                showarrow=False,
                font=dict(color='white', size=12),
                xref='x',
                yref='paper',
                align='center',
                bgcolor=phase_colors[phase].replace('0.15', '0.5'),
                bordercolor='white',
                borderwidth=1,
                borderpad=4
            )
            block_start = row['sequence_counter']
        prev_counter = row['sequence_counter']

    # Add last block
    if block_start is not None:
        fig.add_vrect(
            x0=block_start, x1=prev_counter,
            fillcolor=phase_colors[phase],
            opacity=0.5,
            layer="below",
            line_width=0
        )
        zone_width = prev_counter - block_start + 1
        mid_point = (block_start + prev_counter) / 2
        if zone_width < min_width:
            label_x = block_start + min_width / 2
        else:
            label_x = mid_point
        fig.add_annotation(
            x=label_x,
            y=1.05,
            text=phase,
            showarrow=False,
            font=dict(color='white', size=12),
            xref='x',
            yref='paper',
            align='center',
            bgcolor=phase_colors[phase].replace('0.15', '0.5'),
            bordercolor='white',
            borderwidth=1,
            borderpad=4
        )

fig.show()


In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# Filter one sequence
sequence_id = train_data_df['sequence_id'].iloc[0]
df_seq = train_data_df[train_data_df['sequence_id'] == sequence_id].copy()

# Melt the thm sensor columns
sensor_cols = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']
df_melted = df_seq.melt(
    id_vars=['sequence_counter', 'phase'],
    value_vars=sensor_cols,
    var_name='sensor',
    value_name='value'
)

# Get gesture label for the current sequence (assumes constant within sequence)
gesture_label = df_seq['gesture'].iloc[0]

# Create base line plot with all sensors colored magenta
fig = px.line(
    df_melted,
    x='sequence_counter',
    y='value',
    color='sensor',
    title=f"THM Sensor Readings for Sequence ID {sequence_id} — Gesture: {gesture_label}",
    labels={'sequence_counter': 'Time Step', 'value': 'Sensor Value'},
    template='plotly_dark',
    width=900,
    height=500,
)

# Use magenta shades for thm sensors
magenta_shades = ['#8B008B', '#9400D3', '#9932CC', '#BA55D3', '#DA70D6']
for i, trace in enumerate(fig.data):
    trace.line.color = magenta_shades[i % len(magenta_shades)]

# Colors for phases
phase_colors = {
    'Transition': 'rgba(255,165,0,0.15)',  # orange
    'Pause': 'rgba(0,0,255,0.15)',         # blue
    'Gesture': 'rgba(0,255,0,0.15)'        # green
}

min_width = 5  # Minimum width in timesteps for label placement

# Add shaded phase zones and labels
for phase in df_seq['phase'].unique():
    phase_rows = df_seq[df_seq['phase'] == phase]

    block_start = None
    prev_counter = None

    for _, row in phase_rows.iterrows():
        if block_start is None:
            block_start = row['sequence_counter']
            prev_counter = row['sequence_counter']
        elif row['sequence_counter'] != prev_counter + 1:
            # Add shaded zone
            fig.add_vrect(
                x0=block_start, x1=prev_counter,
                fillcolor=phase_colors[phase],
                opacity=0.3,
                layer="below",
                line_width=0
            )
            # Calculate label x-position with minimum width adjustment
            zone_width = prev_counter - block_start + 1
            mid_point = (block_start + prev_counter) / 2
            if zone_width < min_width:
                label_x = block_start + min_width / 2
            else:
                label_x = mid_point
            # Add label annotation
            fig.add_annotation(
                x=label_x,
                y=1.05,
                text=phase,
                showarrow=False,
                font=dict(color='white', size=12),
                xref='x',
                yref='paper',
                align='center',
                bgcolor=phase_colors[phase].replace('0.15', '0.5'),
                bordercolor='white',
                borderwidth=1,
                borderpad=4
            )
            block_start = row['sequence_counter']
        prev_counter = row['sequence_counter']

    # Add last block
    if block_start is not None:
        fig.add_vrect(
            x0=block_start, x1=prev_counter,
            fillcolor=phase_colors[phase],
            opacity=0.5,
            layer="below",
            line_width=0
        )
        zone_width = prev_counter - block_start + 1
        mid_point = (block_start + prev_counter) / 2
        if zone_width < min_width:
            label_x = block_start + min_width / 2
        else:
            label_x = mid_point
        fig.add_annotation(
            x=label_x,
            y=1.05,
            text=phase,
            showarrow=False,
            font=dict(color='white', size=12),
            xref='x',
            yref='paper',
            align='center',
            bgcolor=phase_colors[phase].replace('0.15', '0.5'),
            bordercolor='white',
            borderwidth=1,
            borderpad=4
        )

fig.show()


In [ ]:
import pandas as pd
import plotly.express as px

# Filter one sequence (you can change sequence_id as needed)
sequence_id = test_data_df['sequence_id'].iloc[0]
df_seq = test_data_df[test_data_df['sequence_id'] == sequence_id].copy()

# Melt the sensor columns to long format for plotting
sensor_cols = ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']
df_melted = df_seq.melt(
    id_vars='sequence_counter',
    value_vars=sensor_cols,
    var_name='sensor',
    value_name='value'
)

# Plot
fig = px.line(
    df_melted,
    x='sequence_counter',
    y='value',
    color='sensor',
    title=f"Sensor Readings for Sequence ID {sequence_id}",
    labels={'sequence_counter': 'Time Step', 'value': 'Sensor Value'},
    template='plotly_dark',
    width=900,
    height=500
)
fig.show()


# IMU GESTURE PREDICTION MODEL

## feature engineering

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R
from sklearn.preprocessing import StandardScaler

def add_imu_tabular_features(df):
    df = df.copy()

    # --- Orientation features ---
    rot_cols = ['rot_x', 'rot_y', 'rot_z', 'rot_w']
    quats = df[rot_cols].astype(float).values

    # Impute missing quaternions with identity quaternion [0,0,0,1]
    invalid_mask = ~np.isfinite(quats).all(axis=1)
    quats[invalid_mask] = [0, 0, 0, 1]

    norms = np.linalg.norm(quats, axis=1)
    zero_norm_mask = norms < 1e-6
    quats[zero_norm_mask] = [0, 0, 0, 1]

    # Normalize quaternions
    norms = np.linalg.norm(quats, axis=1, keepdims=True)
    quats_clean = quats / (norms + 1e-8)

    # Convert to Euler angles
    rotations = R.from_quat(quats_clean)
    eulers = rotations.as_euler('xyz', degrees=False)
    df['roll'] = eulers[:, 0]
    df['pitch'] = eulers[:, 1]
    df['yaw'] = eulers[:, 2]

    # Quaternion magnitude (excluding w)
    df['rot_magnitude'] = np.linalg.norm(quats[:, :3], axis=1)

    # Re-add normalized quaternion components
    df['rot_x_norm'] = quats_clean[:, 0]
    df['rot_y_norm'] = quats_clean[:, 1]
    df['rot_z_norm'] = quats_clean[:, 2]
    df['rot_w_norm'] = quats_clean[:, 3]

    # --- Acceleration features ---
    acc = df[['acc_x', 'acc_y', 'acc_z']].astype(float).values
    acc_x, acc_y, acc_z = acc[:, 0], acc[:, 1], acc[:, 2]

    df['acc_mag'] = np.linalg.norm(acc, axis=1)
    df['acc_xy_angle'] = np.arctan2(acc_y, acc_x)
    df['acc_balance_score'] = np.abs(acc_x - acc_y)
    df['acc_z_ratio'] = acc_z / (df['acc_mag'] + 1e-6)

    # Re-add raw acc
    df['acc_x'] = acc_x
    df['acc_y'] = acc_y
    df['acc_z'] = acc_z

    # --- Anthropometric features ---
    df['height_cm'] = df['height_cm'].astype(float)
    df['shoulder_to_wrist_cm'] = df['shoulder_to_wrist_cm'].astype(float)
    df['elbow_to_wrist_cm'] = df['elbow_to_wrist_cm'].astype(float)

    df['reach_total'] = df['shoulder_to_wrist_cm'] + df['elbow_to_wrist_cm']
    df['reach_ratio'] = df['reach_total'] / (df['height_cm'] + 1e-6)
    df['arm_ratio'] = df['elbow_to_wrist_cm'] / (df['shoulder_to_wrist_cm'] + 1e-6)

    df['age'] = df['age'].astype(float)

    return df


def rotation_features(df):
    new_features = {}

    rot_cols = ['rot_w_mean', 'rot_x_mean', 'rot_y_mean', 'rot_z_mean']
    if all(col in df.columns for col in rot_cols):
        new_features['hand_elevation'] = np.arctan2(df['rot_y_mean'], df['rot_z_mean'])
        new_features['wrist_twist'] = np.arctan2(df['rot_x_mean'], df['rot_w_mean'])

        if all(c in df.columns for c in ['rot_w_std', 'rot_x_std', 'rot_y_std', 'rot_z_std']):
            quat_magnitude_std = np.sqrt(
                df['rot_w_std']**2 + df['rot_x_std']**2 + df['rot_y_std']**2 + df['rot_z_std']**2
            )
            new_features['orientation_stability'] = 1 / (quat_magnitude_std + 1e-8)

    return new_features


def dynamics_features(df):
    new_features = {}

    acc_mean_cols = ['acc_x_mean', 'acc_y_mean', 'acc_z_mean']
    acc_std_cols = ['acc_x_std', 'acc_y_std', 'acc_z_std']

    if all(col in df.columns for col in acc_mean_cols + acc_std_cols):
        acc_magnitude = np.sqrt(df['acc_x_mean']**2 + df['acc_y_mean']**2 + df['acc_z_mean']**2)
        new_features['gesture_intensity'] = acc_magnitude

        acc_jerk = (df['acc_x_std'] + df['acc_y_std'] + df['acc_z_std']) / 3
        new_features['gesture_smoothness'] = 1 / (acc_jerk + 1e-8)

        coord_score = 1 / (np.std([df['acc_x_std'], df['acc_y_std'], df['acc_z_std']], axis=0) + 1e-8)
        new_features['movement_coordination'] = coord_score

    return new_features


def add_summary_imu_features(sequence_stats_df):
    # Add gesture-level engineered features based on summary stats (mean/std columns)
    sequence_stats_df = sequence_stats_df.copy()
    rotation_feats = rotation_features(sequence_stats_df)
    dynamics_feats = dynamics_features(sequence_stats_df)

    for k, v in {**rotation_feats, **dynamics_feats}.items():
        sequence_stats_df[k] = v

    return sequence_stats_df

def extract_gesture_phases(data_df, demo_df, mode = 'train', imu_means = None):
    tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(64)]
    temp_columns = [f'thm_{i}' for i in range(1, 6)]
    tof_temp_columns = tof_columns + temp_columns
    imu_columns = ["acc_x", "acc_y", "acc_z", "rot_w", "rot_x", "rot_y", "rot_z"]
    # get imu only data
    imu_only_data = data_df.drop(columns=tof_temp_columns, errors='ignore')
    # merge with demographics
    imu_merged = pd.merge(imu_only_data, demo_df, on='subject', how='left')

    # filter for phase = gesture
    imu_merged_sorted = imu_merged.sort_values(['sequence_id', 'sequence_counter'])
    if mode == 'train':
      # use phase == gesture for training
      imu_merged_sorted["is_gesture"] = (imu_merged_sorted["phase"] == "Gesture").astype(int)
      gesture_imu_only = imu_merged_sorted[imu_merged_sorted["is_gesture"] == 1]
    if mode == "test":
      # use predicted is_gesture for testing
      gesture_imu_only = imu_merged_sorted[imu_merged_sorted["is_gesture"] == 1]

    # fill any NAs with sequence_id-level means, if not then global means
    # Fill with sequence-level mean
    seq_level_imu_col_means = gesture_imu_only.groupby('sequence_id')[imu_columns].transform('mean')
    gesture_imu_cols_filled = gesture_imu_only[imu_columns].where(~gesture_imu_only[imu_columns].isna(), seq_level_imu_col_means)

    # Fill any remaining with global mean if specified/not specified
    if imu_means is None:
      global_level_imu_col_means = gesture_imu_only[imu_columns].mean()
      gesture_imu_cols_filled = gesture_imu_cols_filled.fillna(global_level_imu_col_means)
    else:
      gesture_imu_cols_filled = gesture_imu_cols_filled.fillna(imu_means)

    non_imu_cols = gesture_imu_only.columns.difference(imu_columns)
    gesture_imu_data = pd.concat([gesture_imu_only[non_imu_cols], gesture_imu_cols_filled], axis=1)

    # add other features
    gesture_imu_data = add_imu_tabular_features(gesture_imu_data)

    # --- Temporal IMU columns ---
    temp_imu_cols = ['roll', 'pitch', 'yaw',
                 'rot_magnitude', 'rot_x_norm', 'rot_y_norm', 'rot_z_norm', 'rot_w_norm',
                 'acc_mag', 'acc_xy_angle', 'acc_balance_score', 'acc_z_ratio',
                 'acc_x', 'acc_y', 'acc_z',
                 'adult_child', 'sex', 'handedness',
                 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm',
                 'reach_total', 'reach_ratio', 'arm_ratio']

    gesture_imu_only_temp_cols = gesture_imu_data[temp_imu_cols]
    cols_to_agg = ["roll", "pitch", 'yaw', 'rot_magnitude', 'rot_x_norm', 'rot_y_norm', 'rot_z_norm', 'rot_w_norm', 'acc_mag', "acc_xy_angle", "acc_balance_score", 'acc_z_ratio', "acc_x","acc_y", "acc_z"]
    gesture_temp_cols_stats = (
    gesture_imu_only_temp_cols[cols_to_agg]
    .groupby(gesture_imu_data.loc[gesture_imu_only_temp_cols.index, 'sequence_id'])
    .agg(['mean', 'min', 'max', 'std']))

    gesture_temp_cols_stats.columns = ['_'.join(col) for col in gesture_temp_cols_stats.columns]
    gesture_temp_cols_stats.reset_index(inplace=True)
    gesture_temp_cols_stats = add_summary_imu_features(gesture_temp_cols_stats)

    imu_with_gesture_stats = pd.merge(gesture_imu_data, gesture_temp_cols_stats, on='sequence_id', how='left')
    imu_with_gesture_stats.drop(columns=['row_id', 'sequence_type', 'subject', 'orientation', 'behavior', 'phase', 'is_gesture'], inplace=True, errors='ignore')

    if imu_means is None:
      return imu_with_gesture_stats, global_level_imu_col_means
    else:
      return imu_with_gesture_stats, imu_means

In [ ]:
extracted_gesture_train_data, global_imu_train_col_means = extract_gesture_phases(train_data_df, train_demo_df, mode = 'train')

In [ ]:
extracted_gesture_test_data, global_imu_test_col_means = extract_gesture_phases(test_data_df, test_demo_df, "test", global_imu_train_col_means)

KeyError: 'is_gesture'

## padding sequences for model

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

def pad_and_reshape(extracted_gesture_data_df, pad_len=80, mode="train", scaler = None):
    df = extracted_gesture_data_df.copy()
    df = df.sort_values(['sequence_id', 'sequence_counter'])

    features = [col for col in df.columns if col not in ['gesture', 'gesture_label', 'sequence_id', 'sequence_counter']]
    if scaler is None:
        scaler = StandardScaler()
        df[features] = scaler.fit_transform(df[features])
    else:
        df[features] = scaler.transform(df[features])
    if mode == 'train':
        # Encode labels
        labelencoder = LabelEncoder()
        df['gesture_label'] = labelencoder.fit_transform(df['gesture'])
        num_classes = len(labelencoder.classes_)
        # Extract and pad sequences
        X_gesture_seqs = [seq_df[features].values for _, seq_df in df.groupby('sequence_id')]
        y_gesture_seqs = [seq_df['gesture_label'].values for _, seq_df in df.groupby('sequence_id')]

        X_padded = pad_sequences(X_gesture_seqs, padding='post', dtype='float32', maxlen=pad_len, value=0.0)
        y_padded = pad_sequences(y_gesture_seqs, padding='post', dtype='int32', maxlen=pad_len, value=num_classes)

        return X_padded, y_padded, labelencoder, scaler

    elif mode == 'test':
        X_gesture_seqs = [seq_df[features].values for _, seq_df in df.groupby('sequence_id')]
        X_padded = pad_sequences(X_gesture_seqs, padding='post', dtype='float32', maxlen=pad_len, value=0.0)

        return X_padded


In [ ]:
X_train_padded_prep, y_train_padded_prep, labelencoder, imu_scaler = pad_and_reshape(extracted_gesture_train_data, pad_len=50, mode="train")

In [ ]:
print(X_train_padded_prep.shape)
print(y_train_padded_prep.shape)

(8150, 50, 92)
(8150, 50)


## custom accuracy and loss for gestures

In [ ]:
padding_class = 18

def masked_accuracy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    # y_true = tf.squeeze(y_true, axis=-1)
    y_pred_labels = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
    mask = tf.not_equal(y_true, padding_class)
    correct = tf.equal(y_true, y_pred_labels)
    correct = tf.logical_and(correct, mask)
    correct = tf.cast(correct, tf.float32)
    mask = tf.cast(mask, tf.float32)
    return tf.reduce_sum(correct) / tf.reduce_sum(mask)

In [ ]:
import tensorflow as tf

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False, reduction='none')

def masked_loss(y_true, y_pred):
    # y_true = tf.squeeze(y_true, axis=-1)  # if needed
    loss = loss_fn(y_true, y_pred)
    mask = tf.cast(tf.not_equal(y_true, padding_class), tf.float32)
    loss = loss * mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)


## IMU only gesture prediction model training

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Masking, TimeDistributed

tf.random.set_seed(0)   # always seed your experiments
Init = tf.keras.initializers.RandomNormal(seed=0)

class StopTrainingOnAccuracy(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs.get('accuracy') is not None and logs.get('masked_accuracy') >= 0.6 and logs.get('val_masked_accuracy') is not None and logs.get('val_accuracy') >= 0.6:
            print("\nReached 60% validation accuracy, stopping training.")
            self.model.stop_training = True

input_dim = X_train_padded_prep.shape[2]
max_seq_len = X_train_padded_prep.shape[1]

inputs = Input(shape=(max_seq_len, input_dim))

# mask padding values
x = Masking(mask_value=0.0)(inputs)

# One or more BiLSTM layers
x = tf.keras.layers.Bidirectional(LSTM(128, return_sequences=True))(x)
x = tf.keras.layers.Dropout(0.4)(x)

x = tf.keras.layers.Bidirectional(LSTM(80, return_sequences=True))(x)
x = tf.keras.layers.Dropout(0.4)(x)

x = tf.keras.layers.Bidirectional(LSTM(60, return_sequences=True))(x)
x = tf.keras.layers.Dropout(0.35)(x)

# Optional dense layer
x = tf.keras.layers.Dense(40, activation='swish')(x)
x = tf.keras.layers.Dropout(0.3)(x)

# Output
outputs = tf.keras.layers.Dense(19, activation='softmax')(x)

# learning rate scheduler
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    0.001,
    decay_steps=1000,
    decay_rate=0.96,
    staircase=True
)

# Nadam optimizer
imu_optimizer = tf.keras.optimizers.Nadam(learning_rate=lr_schedule)

# model
pred_gesture_imu_model = Model(inputs, outputs)

pred_gesture_imu_model.compile(
    optimizer=imu_optimizer,
    loss=masked_loss,
    metrics=[masked_accuracy]
)

# pred_gesture_imu_model.fit(
#    X_train_padded_prep, y_train_padded_prep,
#    validation_split=0.1,
#    # sample_weight=sample_weight,
#    batch_size=40,
#    epochs=100,
#    callbacks=[StopTrainingOnAccuracy()]
# )

Epoch 1/100
184/184 ━━━━━━━━━━━━━━━━━━━━ 13s 34ms/step - loss: 2.3352 - masked_accuracy: 0.2508 - val_loss: 1.6801 - val_masked_accuracy: 0.3874
Epoch 2/100
184/184 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - loss: 1.6902 - masked_accuracy: 0.4134 - val_loss: 1.5430 - val_masked_accuracy: 0.4358
Epoch 3/100
184/184 ━━━━━━━━━━━━━━━━━━━━ 4s 22ms/step - loss: 1.5077 - masked_accuracy: 0.4624 - val_loss: 1.5098 - val_masked_accuracy: 0.4510
Epoch 4/100
184/184 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - loss: 1.4067 - masked_accuracy: 0.4978 - val_loss: 1.4506 - val_masked_accuracy: 0.4613
Epoch 5/100
184/184 ━━━━━━━━━━━━━━━━━━━━ 9s 23ms/step - loss: 1.3079 - masked_accuracy: 0.5247 - val_loss: 1.4345 - val_masked_accuracy: 0.4665
Epoch 6/100
184/184 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - loss: 1.2184 - masked_accuracy: 0.5503 - val_loss: 1.4490 - val_masked_accuracy: 0.4674
Epoch 7/100
184/184 ━━━━━━━━━━━━━━━━━━━━ 9s 23ms/step - loss: 1.1829 - masked_accuracy: 0.5647 - val_loss: 1.4528 - val_masked_accuracy

In [ ]:
# pred_gesture_imu_model.save("pred_gesture_imu_lstm_model.keras")

## loading model

In [ ]:
from tensorflow.keras.models import load_model
pred_gesture_imu_model = load_model('./pred_gesture_imu_lstm_model.keras', custom_objects={
        "masked_loss": masked_loss,
        "masked_accuracy": masked_accuracy,
        "StopTrainingOnAccuracy": StopTrainingOnAccuracy
    })

In [ ]:
train_y_imu_gesture_pred = pred_gesture_imu_model.predict(X_train_padded_prep)


255/255 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step


In [ ]:
train_y_imu_gesture_pred.shape

(8150, 50, 19)

# Total Feature Model, IMU + TEMP + TOF -> GESTURE

## engineering features for full model

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R
import tensorflow as tf
def add_general_tabular_features(df):
    df = df.copy()

    # --- Orientation features ---
    rot_cols = ['rot_x', 'rot_y', 'rot_z', 'rot_w']
    quats = df[rot_cols].astype(float).values

    # Impute missing quaternions with identity quaternion [0,0,0,1]
    invalid_mask = ~np.isfinite(quats).all(axis=1)
    quats[invalid_mask] = [0, 0, 0, 1]

    norms = np.linalg.norm(quats, axis=1)
    zero_norm_mask = norms < 1e-6
    quats[zero_norm_mask] = [0, 0, 0, 1]

    # Normalize quaternions
    norms = np.linalg.norm(quats, axis=1, keepdims=True)
    quats_clean = quats / (norms + 1e-8)

    # Convert to Euler angles
    rotations = R.from_quat(quats_clean)
    eulers = rotations.as_euler('xyz', degrees=False)
    df['roll'] = eulers[:, 0]
    df['pitch'] = eulers[:, 1]
    df['yaw'] = eulers[:, 2]

    # Quaternion magnitude (excluding w)
    df['rot_magnitude'] = np.linalg.norm(quats[:, :3], axis=1)

    # Re-add normalized quaternion components
    df['rot_x_norm'] = quats_clean[:, 0]
    df['rot_y_norm'] = quats_clean[:, 1]
    df['rot_z_norm'] = quats_clean[:, 2]
    df['rot_w_norm'] = quats_clean[:, 3]

    # --- Acceleration features ---
    acc = df[['acc_x', 'acc_y', 'acc_z']].astype(float).values
    acc_x, acc_y, acc_z = acc[:, 0], acc[:, 1], acc[:, 2]

    df['acc_mag'] = np.linalg.norm(acc, axis=1)
    df['acc_xy_angle'] = np.arctan2(acc_y, acc_x)
    df['acc_balance_score'] = np.abs(acc_x - acc_y)
    df['acc_z_ratio'] = acc_z / (df['acc_mag'] + 1e-6)

    # Re-add raw acc
    df['acc_x'] = acc_x
    df['acc_y'] = acc_y
    df['acc_z'] = acc_z

    # --- Anthropometric features ---
    df['height_cm'] = df['height_cm'].astype(float)
    df['shoulder_to_wrist_cm'] = df['shoulder_to_wrist_cm'].astype(float)
    df['elbow_to_wrist_cm'] = df['elbow_to_wrist_cm'].astype(float)

    df['reach_total'] = df['shoulder_to_wrist_cm'] + df['elbow_to_wrist_cm']
    df['reach_ratio'] = df['reach_total'] / (df['height_cm'] + 1e-6)
    df['arm_ratio'] = df['elbow_to_wrist_cm'] / (df['shoulder_to_wrist_cm'] + 1e-6)

    df['age'] = df['age'].astype(float)

    # --- Temperature features ---
    df['tmp_mean'] = df[[f'thm_{i}' for i in range(1, 6)]].mean(axis=1)
    df['tmp_std'] = df[[f'thm_{i}' for i in range(1, 6)]].std(axis=1)
    df['tmp_min'] = df[[f'thm_{i}' for i in range(1, 6)]].min(axis=1)
    df['tmp_max'] = df[[f'thm_{i}' for i in range(1, 6)]].max(axis=1)
    df['tmp_range'] = df['tmp_max'] - df['tmp_min']
    df['tmp_front_mean'] = df[['thm_1', 'thm_2', 'thm_3']].mean(axis=1)
    df['tmp_back_mean'] = df[['thm_4', 'thm_5']].mean(axis=1)
    df['tmp_front_back_diff'] = df['tmp_front_mean'] - df['tmp_back_mean']
    df["tmp_lr_grad"] = df["thm_3"] - df["thm_5"]
    df["tmp_ud_grad"] = df[["thm_1","thm_2"]].mean(axis = 1) - df["thm_4"]

    # --- cross-modal features, temp x imu ---
    df['movement_heat_coupling'] = df["acc_mag"] * df['tmp_range']
    df['lr_movement_temp_sync'] = df["acc_mag"] / (df['tmp_lr_grad'] + 1e-8)
    df['ud_movement_temp_sync'] = df["acc_mag"] / (df['tmp_ud_grad'] + 1e-8)

    return df


def rotation_features(df):
    new_features = {}

    rot_cols = ['rot_w_mean', 'rot_x_mean', 'rot_y_mean', 'rot_z_mean']
    if all(col in df.columns for col in rot_cols):
        new_features['hand_elevation'] = np.arctan2(df['rot_y_mean'], df['rot_z_mean'])
        new_features['wrist_twist'] = np.arctan2(df['rot_x_mean'], df['rot_w_mean'])

        if all(c in df.columns for c in ['rot_w_std', 'rot_x_std', 'rot_y_std', 'rot_z_std']):
            quat_magnitude_std = np.sqrt(
                df['rot_w_std']**2 + df['rot_x_std']**2 + df['rot_y_std']**2 + df['rot_z_std']**2
            )
            new_features['orientation_stability'] = 1 / (quat_magnitude_std + 1e-8)

    return new_features


def dynamics_features(df):
    new_features = {}

    acc_mean_cols = ['acc_x_mean', 'acc_y_mean', 'acc_z_mean']
    acc_std_cols = ['acc_x_std', 'acc_y_std', 'acc_z_std']

    if all(col in df.columns for col in acc_mean_cols + acc_std_cols):
        acc_magnitude = np.sqrt(df['acc_x_mean']**2 + df['acc_y_mean']**2 + df['acc_z_mean']**2)
        new_features['gesture_intensity'] = acc_magnitude

        acc_jerk = (df['acc_x_std'] + df['acc_y_std'] + df['acc_z_std']) / 3
        new_features['gesture_smoothness'] = 1 / (acc_jerk + 1e-8)

        coord_score = 1 / (np.std([df['acc_x_std'], df['acc_y_std'], df['acc_z_std']], axis=0) + 1e-8)
        new_features['movement_coordination'] = coord_score

    return new_features


def add_summary_imu_features(sequence_stats_df):
    # Add gesture-level engineered features based on summary stats (mean/std columns)
    sequence_stats_df = sequence_stats_df.copy()
    rotation_feats = rotation_features(sequence_stats_df)
    dynamics_feats = dynamics_features(sequence_stats_df)

    for k, v in {**rotation_feats, **dynamics_feats}.items():
        sequence_stats_df[k] = v

    return sequence_stats_df

def extract_tof_tensor_data(data_df, demo_df, mode = "train"):
    tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(64)]
    temp_columns = [f'thm_{i}' for i in range(1, 6)]
    tof_temp_columns = tof_columns + temp_columns
    # extract tof_data
    print(len(data_df["sequence_id"].unique()))

    sensor_data_all = data_df.dropna(subset=tof_temp_columns, how='all')

    print(len(sensor_data_all["sequence_id"].unique()))

    # merge with demographics
    sensor_merged = pd.merge(sensor_data_all, demo_df, on='subject', how='left')

    print(len(sensor_merged["sequence_id"].unique()))

    # filter for phase = gesture
    sensor_merged_sorted = sensor_merged.sort_values(['sequence_id', 'sequence_counter'])
    if mode == 'train':
      # use phase == gesture for training
      sensor_merged_sorted["is_gesture"] = (sensor_merged_sorted["phase"] == "Gesture").astype(int)
      gesture_only = sensor_merged_sorted[sensor_merged_sorted["is_gesture"] == 1].copy()
    if mode == "test":
      # use predicted is_gesture for testing
      gesture_only = sensor_merged_sorted[sensor_merged_sorted["is_gesture"] == 1].copy()

    print(len(gesture_only["sequence_id"].unique()))

    tof_data = gesture_only[tof_columns].copy()
    tof_data.fillna(0, inplace = True)
    tof_tensor_list = []
    for seq_id, group in gesture_only.groupby('sequence_id'):
        arr = tof_data.loc[group.index].values
        # Replace -1 with 0
        arr[arr == -1] = 0
        # Normalize by dividing by 254
        arr = arr / 254.0
        # Reshape directly to (timesteps, 5 sensors, 8, 8)
        reshaped = arr.reshape(-1, 5, 8, 8)
        # Optionally move sensor axis to the end → (timesteps, 8, 8, 5)
        reshaped = np.moveaxis(reshaped, 1, -1)
        tof_tensor_list.append(tf.convert_to_tensor(reshaped))

    gesture_only.drop(columns = tof_columns, axis =1, inplace = True)
    return gesture_only, tof_tensor_list


def extract_gesture_phases_full(imu_temp_gesture_df, mode = 'train', imu_temp_sensor_means = None):
    temp_columns = [f'thm_{i}' for i in range(1, 6)]
    imu_columns = ["acc_x", "acc_y", "acc_z", "rot_w", "rot_x", "rot_y", "rot_z"]
    imu_temp_columns = temp_columns + imu_columns
    gesture_only = imu_temp_gesture_df.copy()
    # fill any NAs with sequence_id-level means, if not then global means
    # Fill with sequence-level mean
    seq_level_col_means = gesture_only.groupby('sequence_id')[imu_temp_columns].transform('mean')
    gesture_cols_filled = gesture_only[imu_temp_columns].fillna(seq_level_col_means)

    # Fill any remaining with global mean if specified/not specified
    if imu_temp_sensor_means is None:
      imu_temp_global_col_means = gesture_only[imu_temp_columns].mean()
      gesture_cols_filled = gesture_cols_filled.fillna(imu_temp_global_col_means)
    else:
      gesture_cols_filled = gesture_cols_filled.fillna(imu_temp_sensor_means)

    non_sensor_cols = gesture_only.columns.difference(imu_temp_columns)
    gesture_data = pd.concat([gesture_only[non_sensor_cols], gesture_cols_filled], axis=1)

    # add other features
    gesture_data = add_general_tabular_features(gesture_data)

    cols_to_agg = ["roll", "pitch", 'yaw',
                   'rot_magnitude', 'rot_x_norm', 'rot_y_norm', 'rot_z_norm', 'rot_w_norm',
                   'acc_mag', "acc_xy_angle", "acc_balance_score", 'acc_z_ratio',
                   "acc_x","acc_y", "acc_z",
                   'tmp_mean', 'tmp_std', 'tmp_min', 'tmp_max', 'tmp_range',
                   'tmp_front_mean', 'tmp_back_mean', 'tmp_front_back_diff', 'tmp_lr_grad', 'tmp_ud_grad',
                   'movement_heat_coupling', 'lr_movement_temp_sync', 'ud_movement_temp_sync']

    agg_stats = (
    gesture_data
    .groupby('sequence_id')[cols_to_agg]
    .agg(['mean', 'min', 'max', 'std'])
    )
    agg_stats.columns = [f"{col}_{stat}" for col, stat in agg_stats.columns]
    # add other sequence-level agg stats
    agg_stats = add_summary_imu_features(agg_stats)

    # merge back to timestep level
    gesture_data = gesture_data.merge(agg_stats, on='sequence_id', how='left')

    gesture_data.drop(columns=['row_id', 'sequence_type', 'subject', 'orientation', 'behavior', 'phase', 'is_gesture'], inplace=True, errors='ignore')

    if imu_temp_sensor_means is None:
      return gesture_data, imu_temp_global_col_means
    else:
      return gesture_data, imu_temp_sensor_means

In [ ]:
imu_temp_gesture_df, tof_gesture_tensor = extract_tof_tensor_data(train_data_df, train_demo_df, mode = "train")

8151
8055
8055
8054


In [ ]:
imu_temp_train_data, global_train_col_means = extract_gesture_phases_full(imu_temp_gesture_df, mode = 'train')

In [ ]:
print(imu_temp_train_data.shape)
print(len(tof_gesture_tensor))
print(imu_temp_train_data.isna().sum().sum())
nan_count = 0
for item in tof_gesture_tensor:
    item_array = np.array(item)
    nan_count += np.isnan(item_array).sum()

print(f"Total tof NaNs: {nan_count}")

(252973, 165)
8054
0
Total tof NaNs: 0


## padding sequences for full model, tof tensor

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler


def pad_and_reshape_full(imu_temp_gesture_data, tof_gesture_tensor_list, pad_len=80, mode="train", scaler = None):
    df = imu_temp_gesture_data.copy()
    df = df.sort_values(['sequence_id', 'sequence_counter'])

    features = [col for col in df.columns if col not in ['gesture', 'gesture_label', 'sequence_id', 'sequence_counter']]
    if scaler is None:
        scaler = StandardScaler()
        df[features] = scaler.fit_transform(df[features])
    else:
        df[features] = scaler.transform(df[features])

    # pad tensors to fixed length
    X_tof_padded_list = []
    for tensor in tof_gesture_tensor_list:
        t = tensor.shape[0]
        if t < pad_len:
            padding = tf.zeros((pad_len - t, 8, 8, 5), dtype=tensor.dtype)
            padded = tf.concat([tensor, padding], axis=0)
        else:
            padded = tensor[-pad_len:]
        X_tof_padded_list.append(padded)
    X_tof_padded_arr = np.array(X_tof_padded_list)

    if mode == 'train':
        # Encode labels
        labelencoder = LabelEncoder()
        df = df.copy()
        df['gesture_label'] = labelencoder.fit_transform(df['gesture'])
        num_classes = len(labelencoder.classes_)
        # Extract and pad sequences
        X_gesture_seqs = [seq_df[features].values for _, seq_df in df.groupby('sequence_id')]
        y_gesture_seqs = [seq_df['gesture_label'].values for _, seq_df in df.groupby('sequence_id')]

        X_padded = pad_sequences(X_gesture_seqs, padding='post', dtype='float32', maxlen=pad_len, value=0.0)
        y_padded = pad_sequences(y_gesture_seqs, padding='post', dtype='int32', maxlen=pad_len, value=num_classes)

        return X_padded, X_tof_padded_arr, y_padded, labelencoder, scaler

    elif mode == 'test':
        X_gesture_seqs = [seq_df[features].values for _, seq_df in df.groupby('sequence_id')]
        X_padded = pad_sequences(X_gesture_seqs, padding='post', dtype='float32', maxlen=pad_len, value=0.0)

        return X_padded, X_tof_padded_arr


In [ ]:
X_full_padded, X_tof_padded_arr, y_full_padded, full_labelencoder, full_scaler = pad_and_reshape_full(imu_temp_train_data, tof_gesture_tensor, pad_len = 80, mode = 'train')

In [ ]:
print(X_full_padded.shape)
print(y_full_padded.shape)
print(X_tof_padded_arr.shape)

(8054, 80, 162)
(8054, 80)
(8054, 80, 8, 8, 5)


## custom masked accuracy and loss

In [ ]:
padding_class = 18

def masked_accuracy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    # y_true = tf.squeeze(y_true, axis=-1)
    y_pred_labels = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
    mask = tf.not_equal(y_true, padding_class)
    correct = tf.equal(y_true, y_pred_labels)
    correct = tf.logical_and(correct, mask)
    correct = tf.cast(correct, tf.float32)
    mask = tf.cast(mask, tf.float32)
    return tf.reduce_sum(correct) / tf.reduce_sum(mask)

In [ ]:
import tensorflow as tf

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False, reduction='none')

def masked_loss(y_true, y_pred):
    # y_true = tf.squeeze(y_true, axis=-1)  # if needed
    loss = loss_fn(y_true, y_pred)
    mask = tf.cast(tf.not_equal(y_true, padding_class), tf.float32)
    loss = loss * mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)


## Training full model, 3dconv + LSTM -> gesture

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, Bidirectional, Masking, TimeDistributed, Conv2D, GlobalAveragePooling2D, Concatenate, BatchNormalization

tf.random.set_seed(0)   # always seed your experiments
Init = tf.keras.initializers.RandomNormal(seed=0)

class StopTrainingOnAccuracy(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs.get('accuracy') is not None and logs.get('masked_accuracy') >= 0.75 and logs.get('val_masked_accuracy') is not None and logs.get('val_accuracy') >= 0.75:
            print("\nReached 75% validation accuracy, stopping training.")
            self.model.stop_training = True


# === INPUTS ===
max_seq_len = X_full_padded.shape[1]
imu_temp_input_dim = X_full_padded.shape[2]

imu_input = Input(shape=(max_seq_len, imu_temp_input_dim), name="imu_temp_input")
tof_input = Input(shape=(max_seq_len, 8, 8, 5), name="tof_input")

# === 2D CNN applied on ToF at each timestep ===
tof_features = TimeDistributed(Conv2D(16, (3,3), activation='gelu', padding='same'))(tof_input)
tof_features = TimeDistributed(Conv2D(32, (3,3), activation='gelu', padding='same'))(tof_features)
tof_features = TimeDistributed(GlobalAveragePooling2D())(tof_features)  # (batch, T, 32)

# === Fuse with IMU features ===
fused_features = Concatenate(axis=-1)([imu_input, tof_features])  # (batch, T, input_dim + 32)

# === Mask padded timesteps ===
x = Masking(mask_value=0.0)(fused_features)

# === BiLSTM Stack ===
x = Bidirectional(LSTM(250, return_sequences=True))(x)
x = Dropout(0.4)(x)

x = Bidirectional(LSTM(125, return_sequences=True))(x)
x = Dropout(0.4)(x)

x = Bidirectional(LSTM(80, return_sequences=True))(x)
x = Dropout(0.35)(x)

# === Dense Classifier ===
x = Dense(40, activation='swish')(x)
x = Dropout(0.3)(x)

# === Output layer: sequence-level classification ===
outputs = Dense(19, activation='softmax')(x)  # output shape: (batch, T, num_classes)

# learning rate scheduler
full_lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    0.001,
    decay_steps=1000,
    decay_rate=0.96,
    staircase=True
)

# Nadam optimizer
full_optimizer = tf.keras.optimizers.Nadam(learning_rate=full_lr_schedule)

# model
pred_gesture_full_model = Model([imu_input, tof_input], outputs)

pred_gesture_full_model.compile(
    optimizer=full_optimizer,
    loss=masked_loss,
    metrics=[masked_accuracy]
)

pred_gesture_full_model.fit(
   [X_full_padded, X_tof_padded_arr], y_full_padded,
   validation_split=0.1,
   # sample_weight=sample_weight,
   batch_size=40,
   epochs=50,
   callbacks=[StopTrainingOnAccuracy()]
)

Epoch 1/50
182/182 ━━━━━━━━━━━━━━━━━━━━ 81s 222ms/step - loss: 2.0041 - masked_accuracy: 0.3597 - val_loss: 1.3840 - val_masked_accuracy: 0.4806
Epoch 2/50
182/182 ━━━━━━━━━━━━━━━━━━━━ 62s 132ms/step - loss: 1.2888 - masked_accuracy: 0.5317 - val_loss: 1.2579 - val_masked_accuracy: 0.5361
Epoch 3/50
182/182 ━━━━━━━━━━━━━━━━━━━━ 41s 133ms/step - loss: 1.1279 - masked_accuracy: 0.5780 - val_loss: 1.1558 - val_masked_accuracy: 0.5571
Epoch 4/50
182/182 ━━━━━━━━━━━━━━━━━━━━ 40s 126ms/step - loss: 0.9964 - masked_accuracy: 0.6214 - val_loss: 1.1290 - val_masked_accuracy: 0.5708
Epoch 5/50
182/182 ━━━━━━━━━━━━━━━━━━━━ 41s 126ms/step - loss: 0.9289 - masked_accuracy: 0.6488 - val_loss: 1.1184 - val_masked_accuracy: 0.6073
Epoch 6/50
182/182 ━━━━━━━━━━━━━━━━━━━━ 42s 134ms/step - loss: 0.9039 - masked_accuracy: 0.6625 - val_loss: 1.1129 - val_masked_accuracy: 0.6136
Epoch 7/50
182/182 ━━━━━━━━━━━━━━━━━━━━ 49s 176ms/step - loss: 0.7916 - masked_accuracy: 0.6944 - val_loss: 1.0514 - val_masked_ac

In [ ]:
pred_gesture_full_model.save("pred_gesture_full_lstm_cnn_model.keras")

# IMU ONLY PHASE == GESTURE PREDICTION

In [ ]:
from scipy.spatial.transform import Rotation as R
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder


def add_imu_tabular_features(df):
    df = df.copy()

    # --- Orientation features ---
    rot_cols = ['rot_x', 'rot_y', 'rot_z', 'rot_w']
    quats = df[rot_cols].astype(float).values

    # Impute missing quaternions with identity quaternion [0,0,0,1]
    invalid_mask = ~np.isfinite(quats).all(axis=1)
    quats[invalid_mask] = [0, 0, 0, 1]

    norms = np.linalg.norm(quats, axis=1)
    zero_norm_mask = norms < 1e-6
    quats[zero_norm_mask] = [0, 0, 0, 1]

    # Normalize quaternions
    norms = np.linalg.norm(quats, axis=1, keepdims=True)
    quats_clean = quats / (norms + 1e-8)

    # Convert to Euler angles
    rotations = R.from_quat(quats_clean)
    eulers = rotations.as_euler('xyz', degrees=False)
    df['roll'] = eulers[:, 0]
    df['pitch'] = eulers[:, 1]
    df['yaw'] = eulers[:, 2]

    # Quaternion magnitude (excluding w)
    df['rot_magnitude'] = np.linalg.norm(quats[:, :3], axis=1)

    # Re-add normalized quaternion components
    df['rot_x_norm'] = quats_clean[:, 0]
    df['rot_y_norm'] = quats_clean[:, 1]
    df['rot_z_norm'] = quats_clean[:, 2]
    df['rot_w_norm'] = quats_clean[:, 3]

    # --- Acceleration features ---
    acc = df[['acc_x', 'acc_y', 'acc_z']].astype(float).values
    acc_x, acc_y, acc_z = acc[:, 0], acc[:, 1], acc[:, 2]

    df['acc_mag'] = np.linalg.norm(acc, axis=1)
    df['acc_xy_angle'] = np.arctan2(acc_y, acc_x)
    df['acc_balance_score'] = np.abs(acc_x - acc_y)
    df['acc_z_ratio'] = acc_z / (df['acc_mag'] + 1e-6)

    # Re-add raw acc
    df['acc_x'] = acc_x
    df['acc_y'] = acc_y
    df['acc_z'] = acc_z

    # --- Anthropometric features ---
    df['height_cm'] = df['height_cm'].astype(float)
    df['shoulder_to_wrist_cm'] = df['shoulder_to_wrist_cm'].astype(float)
    df['elbow_to_wrist_cm'] = df['elbow_to_wrist_cm'].astype(float)

    df['reach_total'] = df['shoulder_to_wrist_cm'] + df['elbow_to_wrist_cm']
    df['reach_ratio'] = df['reach_total'] / (df['height_cm'] + 1e-6)
    df['arm_ratio'] = df['elbow_to_wrist_cm'] / (df['shoulder_to_wrist_cm'] + 1e-6)

    df['age'] = df['age'].astype(float)

    return df

# pipeline for training/test preprocessing
def imu_sequence_pipeline(data_df, demo_df, max_seq_len=700, mode='train', train_means = None):
    assert mode in ['train', 'test'], "mode must be either 'train' or 'test'"

    # --- Remove tof and temp columns ---
    tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(64)]
    temp_columns = [f'thm_{i}' for i in range(1, 6)]
    tof_temp_columns = tof_columns + temp_columns
    imu_only_data = data_df.drop(columns=tof_temp_columns, errors='ignore')

    # --- Merge with demographics ---
    imu_merged = pd.merge(imu_only_data, demo_df, on='subject', how='left')

    # --- Feature engineering ---
    imu_merged = add_imu_tabular_features(imu_merged)

    # --- Temporal IMU columns ---
    temp_imu_cols = ['roll', 'pitch', 'yaw',
                 'rot_magnitude', 'rot_x_norm', 'rot_y_norm', 'rot_z_norm', 'rot_w_norm',
                 'acc_mag', 'acc_xy_angle', 'acc_balance_score', 'acc_z_ratio',
                 'acc_x', 'acc_y', 'acc_z',
                 'adult_child', 'sex', 'handedness',
                 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm',
                 'reach_ratio', 'arm_ratio']

    if mode == 'train':
        train_means = imu_merged[temp_imu_cols].mean()
        imu_merged[temp_imu_cols] = imu_merged[temp_imu_cols].fillna(train_means)
    else:  # test
        if train_means is not None:
            imu_merged[temp_imu_cols] = imu_merged[temp_imu_cols].fillna(train_means)
        else:
            raise ValueError("Must provide train_means during test mode")

    # --- Sort and group ---
    imu_merged_sorted = imu_merged.sort_values(['sequence_id', 'sequence_counter'])
    grouped = imu_merged_sorted.groupby('sequence_id')[temp_imu_cols]

    # Extract list of sequences
    X_seqs = grouped.apply(lambda x: x.values).tolist()
    X_padded = pad_sequences(X_seqs, maxlen=max_seq_len, dtype='float32', padding='post', truncating='pre', value=0.0)

    if mode == 'test':
        if len(X_padded.shape) == 2:
            X_padded = np.expand_dims(X_padded, axis=0)
        return X_padded

    # --- Only for training: create labels for gesture phase ---
    imu_merged_sorted['is_gesture'] = (imu_merged_sorted['phase'] == 'Gesture').astype(int)
    y_seqs = imu_merged_sorted.groupby('sequence_id')['is_gesture'].apply(lambda x: x.values).tolist()
    y_phase_padded = pad_sequences(y_seqs, maxlen=max_seq_len, dtype='int32', padding='post', truncating='pre', value=0)
    y_phase_padded = y_phase_padded[..., np.newaxis]

    return X_padded, y_phase_padded, train_means

In [ ]:
X_imu_phase_train, y_imu_phase_train, imu_phase_train_means = imu_sequence_pipeline(
    max_seq_len=700,
    data_df=train_data_df,
    demo_df=train_demo_df,
    mode='train'
)
print("train X shape:", X_imu_phase_train.shape)
print("train phase y shape:", y_imu_phase_train.shape)

X_imu_phase_test = imu_sequence_pipeline(
    max_seq_len=700,
    data_df=test_data_df,
    demo_df=test_demo_df,
    mode='test',
    train_means=imu_phase_train_means)
print("test X shape", X_imu_phase_test.shape)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Masking, TimeDistributed

tf.random.set_seed(0)   # always seed your experiments
Init = tf.keras.initializers.RandomNormal(seed=0)

input_dim = X_imu_phase_train.shape[2]
max_seq_len = X_imu_phase_train.shape[1]

inputs = Input(shape=(max_seq_len, input_dim))

# mask padding values
# x = Masking(mask_value=0.0)(inputs)

# One or more BiLSTM layers
x = tf.keras.layers.Bidirectional(LSTM(128, return_sequences=True))(inputs)
x = tf.keras.layers.Dropout(0.3)(x)

x = tf.keras.layers.Bidirectional(LSTM(80, return_sequences=True))(x)
x = tf.keras.layers.Dropout(0.3)(x)

# Optional dense layer
x = tf.keras.layers.Dense(40, activation='gelu')(x)
x = tf.keras.layers.Dropout(0.3)(x)

# Output
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)


# learning rate scheduler
imu_phase_lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    0.001,
    decay_steps=500,
    decay_rate=0.96,
    staircase=True
)

# Nadam optimizer
imu_phase_optimizer = tf.keras.optimizers.Nadam(learning_rate=imu_phase_lr_schedule)

pred_phase_imu_model = Model(inputs, outputs)

pred_phase_imu_model.compile(
    optimizer=imu_phase_optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# pred_phase_imu_model.fit(X_imu_phase_train, y_imu_phase_train, batch_size=100, epochs=10, validation_split=0.1)

In [ ]:
# pred_phase_imu_model.save("pred_phase_imu_lstm_model.keras")

In [ ]:
from tensorflow.keras.models import load_model
pred_phase_imu_model = load_model('./pred_phase_imu_lstm_model.keras')

In [ ]:
train_y_imu_phase_pred = pred_phase_imu_model.predict(X_imu_phase_train)
train_y_imu_phase_pred_binary = (train_y_imu_phase_pred > 0.5).astype(int)

from sklearn.metrics import classification_report

print(classification_report(y_imu_phase_train.flatten(), train_y_imu_phase_pred_binary.flatten(), digits=3))

255/255 ━━━━━━━━━━━━━━━━━━━━ 12s 41ms/step
              precision    recall  f1-score   support

           0      0.998     0.999     0.999   5449883
           1      0.975     0.965     0.970    255817

    accuracy                          0.997   5705700
   macro avg      0.987     0.982     0.984   5705700
weighted avg      0.997     0.997     0.997   5705700



In [ ]:
def reshape_gesture_pred_per_seq(data_df, train_y_imu_phase_pred):
    train_y_imu_phase_pred_binary = (train_y_imu_phase_pred > 0.5).astype(int)
    pred_flat = []

    max_timesteps_per_seq = data_df.groupby("sequence_id")["sequence_counter"].max()

    for i, seq_id in enumerate(max_timesteps_per_seq.index):
        seq_len = max_timesteps_per_seq.loc[seq_id] + 1
        probs = train_y_imu_phase_pred[i][:seq_len].squeeze()
        binary = (probs > 0.5).astype(int)

        # Enforce at least one gesture per sequence
        if binary.sum() == 0:
            binary[np.argmax(probs)] = 1

        pred_flat.extend(binary.tolist())
    assert len(pred_flat) == len(data_df), f"Length mismatch: {len(pred_flat)} != {len(data_df)}"

    data_df = data_df.copy()
    data_df["is_gesture"] = pred_flat
    return data_df

# PREDICT CMI PIPELINE


In [ ]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(0)   # always seed your experiments
Init = tf.keras.initializers.RandomNormal(seed=0)

### IMU Phase Model

In [ ]:
from scipy.spatial.transform import Rotation as R
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder


def add_imu_tabular_features(df):
    df = df.copy()

    # --- Orientation features ---
    rot_cols = ['rot_x', 'rot_y', 'rot_z', 'rot_w']
    quats = df[rot_cols].astype(float).values

    # Impute missing quaternions with identity quaternion [0,0,0,1]
    invalid_mask = ~np.isfinite(quats).all(axis=1)
    quats[invalid_mask] = [0, 0, 0, 1]

    norms = np.linalg.norm(quats, axis=1)
    zero_norm_mask = norms < 1e-6
    quats[zero_norm_mask] = [0, 0, 0, 1]

    # Normalize quaternions
    norms = np.linalg.norm(quats, axis=1, keepdims=True)
    quats_clean = quats / (norms + 1e-8)

    # Convert to Euler angles
    rotations = R.from_quat(quats_clean)
    eulers = rotations.as_euler('xyz', degrees=False)
    df['roll'] = eulers[:, 0]
    df['pitch'] = eulers[:, 1]
    df['yaw'] = eulers[:, 2]

    # Quaternion magnitude (excluding w)
    df['rot_magnitude'] = np.linalg.norm(quats[:, :3], axis=1)

    # Re-add normalized quaternion components
    df['rot_x_norm'] = quats_clean[:, 0]
    df['rot_y_norm'] = quats_clean[:, 1]
    df['rot_z_norm'] = quats_clean[:, 2]
    df['rot_w_norm'] = quats_clean[:, 3]

    # --- Acceleration features ---
    acc = df[['acc_x', 'acc_y', 'acc_z']].astype(float).values
    acc_x, acc_y, acc_z = acc[:, 0], acc[:, 1], acc[:, 2]

    df['acc_mag'] = np.linalg.norm(acc, axis=1)
    df['acc_xy_angle'] = np.arctan2(acc_y, acc_x)
    df['acc_balance_score'] = np.abs(acc_x - acc_y)
    df['acc_z_ratio'] = acc_z / (df['acc_mag'] + 1e-6)

    # Re-add raw acc
    df['acc_x'] = acc_x
    df['acc_y'] = acc_y
    df['acc_z'] = acc_z

    # --- Anthropometric features ---
    df['height_cm'] = df['height_cm'].astype(float)
    df['shoulder_to_wrist_cm'] = df['shoulder_to_wrist_cm'].astype(float)
    df['elbow_to_wrist_cm'] = df['elbow_to_wrist_cm'].astype(float)

    df['reach_total'] = df['shoulder_to_wrist_cm'] + df['elbow_to_wrist_cm']
    df['reach_ratio'] = df['reach_total'] / (df['height_cm'] + 1e-6)
    df['arm_ratio'] = df['elbow_to_wrist_cm'] / (df['shoulder_to_wrist_cm'] + 1e-6)

    df['age'] = df['age'].astype(float)

    return df

# pipeline for training/test preprocessing
def imu_sequence_pipeline(data_df, demo_df, max_seq_len=700, mode='train', train_means = None):
    assert mode in ['train', 'test'], "mode must be either 'train' or 'test'"

    # --- Remove tof and temp columns ---
    tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(64)]
    temp_columns = [f'thm_{i}' for i in range(1, 6)]
    tof_temp_columns = tof_columns + temp_columns
    imu_only_data = data_df.drop(columns=tof_temp_columns, errors='ignore')

    # --- Merge with demographics ---
    imu_merged = pd.merge(imu_only_data, demo_df, on='subject', how='left')

    # --- Feature engineering ---
    imu_merged = add_imu_tabular_features(imu_merged)

    # --- Temporal IMU columns ---
    temp_imu_cols = ['roll', 'pitch', 'yaw',
                 'rot_magnitude', 'rot_x_norm', 'rot_y_norm', 'rot_z_norm', 'rot_w_norm',
                 'acc_mag', 'acc_xy_angle', 'acc_balance_score', 'acc_z_ratio',
                 'acc_x', 'acc_y', 'acc_z',
                 'adult_child', 'sex', 'handedness',
                 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm',
                 'reach_ratio', 'arm_ratio']

    if mode == 'train':
        train_means = imu_merged[temp_imu_cols].mean()
        imu_merged[temp_imu_cols] = imu_merged[temp_imu_cols].fillna(train_means)
    else:  # test
        if train_means is not None:
            imu_merged[temp_imu_cols] = imu_merged[temp_imu_cols].fillna(train_means)
        else:
            raise ValueError("Must provide train_means during test mode")

    # --- Sort and group ---
    imu_merged_sorted = imu_merged.sort_values(['sequence_id', 'sequence_counter'])
    grouped = imu_merged_sorted.groupby('sequence_id')[temp_imu_cols]

    # Extract list of sequences
    X_seqs = grouped.apply(lambda x: x.values).tolist()
    X_padded = pad_sequences(X_seqs, maxlen=max_seq_len, dtype='float32', padding='post', truncating='pre', value=0.0)

    if mode == 'test':
        if len(X_padded.shape) == 2:
            X_padded = np.expand_dims(X_padded, axis=0)
        return X_padded

    # --- Only for training: create labels for gesture phase ---
    imu_merged_sorted['is_gesture'] = (imu_merged_sorted['phase'] == 'Gesture').astype(int)
    y_seqs = imu_merged_sorted.groupby('sequence_id')['is_gesture'].apply(lambda x: x.values).tolist()
    y_phase_padded = pad_sequences(y_seqs, maxlen=max_seq_len, dtype='int32', padding='post', truncating='pre', value=0)
    y_phase_padded = y_phase_padded[..., np.newaxis]

    return X_padded, y_phase_padded, train_means

In [ ]:
_, _, imu_phase_train_means = imu_sequence_pipeline(
    max_seq_len=700,
    data_df=train_data_df,
    demo_df=train_demo_df,
    mode='train'
)

train X shape: (8151, 700, 23)
train phase y shape: (8151, 700, 1)
test X shape (2, 700, 23)


In [ ]:
from tensorflow.keras.models import load_model
pred_phase_imu_model = load_model('./pred_phase_imu_lstm_model.keras')

In [ ]:
def reshape_gesture_pred_per_seq(data_df, train_y_imu_phase_pred, top_k=4):
    pred_flat = []

    max_timesteps_per_seq = data_df.groupby("sequence_id")["sequence_counter"].max()

    for i, seq_id in enumerate(max_timesteps_per_seq.index):
        seq_len = max_timesteps_per_seq.loc[seq_id] + 1
        probs = train_y_imu_phase_pred[i][:seq_len].squeeze()

        # Threshold to get binary predictions
        binary = (probs > 0.5).astype(int)

        # Enforce at least top_k gestures per sequence
        if binary.sum() < top_k:
            top_k_indices = np.argsort(probs)[-top_k:]  # Get indices of top-k probs
            binary[:] = 0  # Reset all to 0
            binary[top_k_indices] = 1

        pred_flat.extend(binary.tolist())

    assert len(pred_flat) == len(data_df), f"Length mismatch: {len(pred_flat)} != {len(data_df)}"

    data_df = data_df.copy()
    data_df["is_gesture"] = pred_flat
    return data_df

### IMU ONLY GESTURE MODEL

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R
from sklearn.preprocessing import StandardScaler

def rotation_features(df):
    new_features = {}

    rot_cols = ['rot_w_mean', 'rot_x_mean', 'rot_y_mean', 'rot_z_mean']
    if all(col in df.columns for col in rot_cols):
        new_features['hand_elevation'] = np.arctan2(df['rot_y_mean'], df['rot_z_mean'])
        new_features['wrist_twist'] = np.arctan2(df['rot_x_mean'], df['rot_w_mean'])

        if all(c in df.columns for c in ['rot_w_std', 'rot_x_std', 'rot_y_std', 'rot_z_std']):
            quat_magnitude_std = np.sqrt(
                df['rot_w_std']**2 + df['rot_x_std']**2 + df['rot_y_std']**2 + df['rot_z_std']**2
            )
            new_features['orientation_stability'] = 1 / (quat_magnitude_std + 1e-8)

    return new_features


def dynamics_features(df):
    new_features = {}

    acc_mean_cols = ['acc_x_mean', 'acc_y_mean', 'acc_z_mean']
    acc_std_cols = ['acc_x_std', 'acc_y_std', 'acc_z_std']

    if all(col in df.columns for col in acc_mean_cols + acc_std_cols):
        acc_magnitude = np.sqrt(df['acc_x_mean']**2 + df['acc_y_mean']**2 + df['acc_z_mean']**2)
        new_features['gesture_intensity'] = acc_magnitude

        acc_jerk = (df['acc_x_std'] + df['acc_y_std'] + df['acc_z_std']) / 3
        new_features['gesture_smoothness'] = 1 / (acc_jerk + 1e-8)

        coord_score = 1 / (np.std([df['acc_x_std'], df['acc_y_std'], df['acc_z_std']], axis=0) + 1e-8)
        new_features['movement_coordination'] = coord_score

    return new_features


def add_summary_imu_features(sequence_stats_df):
    # Add gesture-level engineered features based on summary stats (mean/std columns)
    sequence_stats_df = sequence_stats_df.copy()
    rotation_feats = rotation_features(sequence_stats_df)
    dynamics_feats = dynamics_features(sequence_stats_df)

    for k, v in {**rotation_feats, **dynamics_feats}.items():
        sequence_stats_df[k] = v

    return sequence_stats_df

def extract_gesture_phases(data_df, demo_df, mode = 'train', imu_means = None):
    tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(64)]
    temp_columns = [f'thm_{i}' for i in range(1, 6)]
    tof_temp_columns = tof_columns + temp_columns
    imu_columns = ["acc_x", "acc_y", "acc_z", "rot_w", "rot_x", "rot_y", "rot_z"]
    # get imu only data
    imu_only_data = data_df.drop(columns=tof_temp_columns, errors='ignore')
    # merge with demographics
    imu_merged = pd.merge(imu_only_data, demo_df, on='subject', how='left')

    # filter for phase = gesture
    imu_merged_sorted = imu_merged.sort_values(['sequence_id', 'sequence_counter'])
    if mode == 'train':
      # use phase == gesture for training
      imu_merged_sorted["is_gesture"] = (imu_merged_sorted["phase"] == "Gesture").astype(int)
      gesture_imu_only = imu_merged_sorted[imu_merged_sorted["is_gesture"] == 1]
    if mode == "test":
      # use predicted is_gesture for testing
      gesture_imu_only = imu_merged_sorted[imu_merged_sorted["is_gesture"] == 1]

    # fill any NAs with sequence_id-level means, if not then global means
    # Fill with sequence-level mean
    seq_level_imu_col_means = gesture_imu_only.groupby('sequence_id')[imu_columns].transform('mean')
    gesture_imu_cols_filled = gesture_imu_only[imu_columns].where(~gesture_imu_only[imu_columns].isna(), seq_level_imu_col_means)

    # Fill any remaining with global mean if specified/not specified
    if imu_means is None:
      global_level_imu_col_means = gesture_imu_only[imu_columns].mean()
      gesture_imu_cols_filled = gesture_imu_cols_filled.fillna(global_level_imu_col_means)
    else:
      gesture_imu_cols_filled = gesture_imu_cols_filled.fillna(imu_means)

    non_imu_cols = gesture_imu_only.columns.difference(imu_columns)
    gesture_imu_data = pd.concat([gesture_imu_only[non_imu_cols], gesture_imu_cols_filled], axis=1)

    # add other features
    gesture_imu_data = add_imu_tabular_features(gesture_imu_data)

    # --- Temporal IMU columns ---
    temp_imu_cols = ['roll', 'pitch', 'yaw',
                 'rot_magnitude', 'rot_x_norm', 'rot_y_norm', 'rot_z_norm', 'rot_w_norm',
                 'acc_mag', 'acc_xy_angle', 'acc_balance_score', 'acc_z_ratio',
                 'acc_x', 'acc_y', 'acc_z',
                 'adult_child', 'sex', 'handedness',
                 'height_cm', 'shoulder_to_wrist_cm', 'elbow_to_wrist_cm',
                 'reach_total', 'reach_ratio', 'arm_ratio']

    gesture_imu_only_temp_cols = gesture_imu_data[temp_imu_cols]
    cols_to_agg = ["roll", "pitch", 'yaw', 'rot_magnitude', 'rot_x_norm', 'rot_y_norm', 'rot_z_norm', 'rot_w_norm', 'acc_mag', "acc_xy_angle", "acc_balance_score", 'acc_z_ratio', "acc_x","acc_y", "acc_z"]
    gesture_temp_cols_stats = (
    gesture_imu_only_temp_cols[cols_to_agg]
    .groupby(gesture_imu_data.loc[gesture_imu_only_temp_cols.index, 'sequence_id'])
    .agg(['mean', 'min', 'max', 'std']))

    gesture_temp_cols_stats.columns = ['_'.join(col) for col in gesture_temp_cols_stats.columns]
    gesture_temp_cols_stats.reset_index(inplace=True)
    gesture_temp_cols_stats = add_summary_imu_features(gesture_temp_cols_stats)

    imu_with_gesture_stats = pd.merge(gesture_imu_data, gesture_temp_cols_stats, on='sequence_id', how='left')
    imu_with_gesture_stats.drop(columns=['row_id', 'sequence_type', 'subject', 'orientation', 'behavior', 'phase', 'is_gesture'], inplace=True, errors='ignore')

    if imu_means is None:
      return imu_with_gesture_stats, global_level_imu_col_means
    else:
      return imu_with_gesture_stats, imu_means

In [ ]:
extracted_gesture_imu_train_data, global_imu_train_col_means = extract_gesture_phases(train_data_df, train_demo_df, mode = 'train')

## padding sequences for model

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

def pad_and_reshape(extracted_gesture_data_df, pad_len=80, mode="train", scaler = None):
    df = extracted_gesture_data_df.copy()
    df = df.sort_values(['sequence_id', 'sequence_counter'])

    features = [col for col in df.columns if col not in ['gesture', 'gesture_label', 'sequence_id', 'sequence_counter']]
    if scaler is None:
        scaler = StandardScaler()
        df[features] = scaler.fit_transform(df[features])
    else:
        df[features] = scaler.transform(df[features])
    if mode == 'train':
        # Encode labels
        labelencoder = LabelEncoder()
        df['gesture_label'] = labelencoder.fit_transform(df['gesture'])
        num_classes = len(labelencoder.classes_)
        # Extract and pad sequences
        X_gesture_seqs = [seq_df[features].values for _, seq_df in df.groupby('sequence_id')]
        y_gesture_seqs = [seq_df['gesture_label'].values for _, seq_df in df.groupby('sequence_id')]

        X_padded = pad_sequences(X_gesture_seqs, padding='post', dtype='float32', maxlen=pad_len, value=0.0)
        y_padded = pad_sequences(y_gesture_seqs, padding='post', dtype='int32', maxlen=pad_len, value=num_classes)

        return X_padded, y_padded, labelencoder, scaler

    elif mode == 'test':
        X_gesture_seqs = [seq_df[features].values for _, seq_df in df.groupby('sequence_id')]
        X_padded = pad_sequences(X_gesture_seqs, padding='post', dtype='float32', maxlen=pad_len, value=0.0)

        return X_padded


## custom accuracy and loss for gestures

In [ ]:
_, _, imu_labelencoder, imu_scaler = pad_and_reshape(extracted_gesture_imu_train_data, pad_len=50, mode="train")
del extracted_gesture_imu_train_data

In [ ]:
import tensorflow as tf

class StopTrainingOnAccuracy(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs.get('accuracy') is not None and logs.get('masked_accuracy') >= 0.75 and logs.get('val_masked_accuracy') is not None and logs.get('val_accuracy') >= 0.75:
            print("\nReached 75% validation accuracy, stopping training.")
            self.model.stop_training = True

padding_class = 18

def masked_accuracy(y_true, y_pred):
    y_true = tf.cast(y_true, tf.int32)
    # y_true = tf.squeeze(y_true, axis=-1)
    y_pred_labels = tf.argmax(y_pred, axis=-1, output_type=tf.int32)
    mask = tf.not_equal(y_true, padding_class)
    correct = tf.equal(y_true, y_pred_labels)
    correct = tf.logical_and(correct, mask)
    correct = tf.cast(correct, tf.float32)
    mask = tf.cast(mask, tf.float32)
    return tf.reduce_sum(correct) / tf.reduce_sum(mask)


loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False, reduction='none')

def masked_loss(y_true, y_pred):
    # y_true = tf.squeeze(y_true, axis=-1)  # if needed
    loss = loss_fn(y_true, y_pred)
    mask = tf.cast(tf.not_equal(y_true, padding_class), tf.float32)
    loss = loss * mask
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

## loading model

In [ ]:
from tensorflow.keras.models import load_model
pred_gesture_imu_model = load_model('./pred_gesture_imu_lstm_model.keras', custom_objects={
        "masked_loss": masked_loss,
        "masked_accuracy": masked_accuracy,
        "StopTrainingOnAccuracy": StopTrainingOnAccuracy
    })

In [ ]:
import numpy as np
from scipy.stats import mode

# Assume `preds` is your (8150, 50, 19) array of logits or probabilities
# preds = model.predict(...)  # for example

def sequence_mode_prediction(preds):
    # Step 1: Argmax across class axis
    top2 = np.argpartition(-preds, kth=1, axis=2)[..., :2]  # get top-2 class indices
    top_scores = np.take_along_axis(preds, top2, axis=2)   # get their scores

    # Sort top-2 by actual score
    sorted_idx = np.argsort(-top_scores, axis=2)  # sort descending
    sorted_top2 = np.take_along_axis(top2, sorted_idx, axis=2)  # reorder top2 by score

    # Step 2: Replace argmax=18 (padding) with next highest
    pred_classes = sorted_top2[..., 0]
    second_best = sorted_top2[..., 1]
    mask_pad = pred_classes == 18
    pred_classes[mask_pad] = second_best[mask_pad]  # replace pad preds with next-best

    # Step 3: Sequence-wise mode
    mode_preds, _ = mode(pred_classes, axis=1, keepdims=False)

    return mode_preds

### FULL GESTURE MODEL

In [ ]:
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R
import tensorflow as tf
def add_general_tabular_features(df):
    df = df.copy()

    # --- Orientation features ---
    rot_cols = ['rot_x', 'rot_y', 'rot_z', 'rot_w']
    quats = df[rot_cols].astype(float).values

    # Impute missing quaternions with identity quaternion [0,0,0,1]
    invalid_mask = ~np.isfinite(quats).all(axis=1)
    quats[invalid_mask] = [0, 0, 0, 1]

    norms = np.linalg.norm(quats, axis=1)
    zero_norm_mask = norms < 1e-6
    quats[zero_norm_mask] = [0, 0, 0, 1]

    # Normalize quaternions
    norms = np.linalg.norm(quats, axis=1, keepdims=True)
    quats_clean = quats / (norms + 1e-8)

    # Convert to Euler angles
    rotations = R.from_quat(quats_clean)
    eulers = rotations.as_euler('xyz', degrees=False)
    df['roll'] = eulers[:, 0]
    df['pitch'] = eulers[:, 1]
    df['yaw'] = eulers[:, 2]

    # Quaternion magnitude (excluding w)
    df['rot_magnitude'] = np.linalg.norm(quats[:, :3], axis=1)

    # Re-add normalized quaternion components
    df['rot_x_norm'] = quats_clean[:, 0]
    df['rot_y_norm'] = quats_clean[:, 1]
    df['rot_z_norm'] = quats_clean[:, 2]
    df['rot_w_norm'] = quats_clean[:, 3]

    # --- Acceleration features ---
    acc = df[['acc_x', 'acc_y', 'acc_z']].astype(float).values
    acc_x, acc_y, acc_z = acc[:, 0], acc[:, 1], acc[:, 2]

    df['acc_mag'] = np.linalg.norm(acc, axis=1)
    df['acc_xy_angle'] = np.arctan2(acc_y, acc_x)
    df['acc_balance_score'] = np.abs(acc_x - acc_y)
    df['acc_z_ratio'] = acc_z / (df['acc_mag'] + 1e-6)

    # Re-add raw acc
    df['acc_x'] = acc_x
    df['acc_y'] = acc_y
    df['acc_z'] = acc_z

    # --- Anthropometric features ---
    df['height_cm'] = df['height_cm'].astype(float)
    df['shoulder_to_wrist_cm'] = df['shoulder_to_wrist_cm'].astype(float)
    df['elbow_to_wrist_cm'] = df['elbow_to_wrist_cm'].astype(float)

    df['reach_total'] = df['shoulder_to_wrist_cm'] + df['elbow_to_wrist_cm']
    df['reach_ratio'] = df['reach_total'] / (df['height_cm'] + 1e-6)
    df['arm_ratio'] = df['elbow_to_wrist_cm'] / (df['shoulder_to_wrist_cm'] + 1e-6)

    df['age'] = df['age'].astype(float)

    # --- Temperature features ---
    df['tmp_mean'] = df[[f'thm_{i}' for i in range(1, 6)]].mean(axis=1)
    df['tmp_std'] = df[[f'thm_{i}' for i in range(1, 6)]].std(axis=1)
    df['tmp_min'] = df[[f'thm_{i}' for i in range(1, 6)]].min(axis=1)
    df['tmp_max'] = df[[f'thm_{i}' for i in range(1, 6)]].max(axis=1)
    df['tmp_range'] = df['tmp_max'] - df['tmp_min']
    df['tmp_front_mean'] = df[['thm_1', 'thm_2', 'thm_3']].mean(axis=1)
    df['tmp_back_mean'] = df[['thm_4', 'thm_5']].mean(axis=1)
    df['tmp_front_back_diff'] = df['tmp_front_mean'] - df['tmp_back_mean']
    df["tmp_lr_grad"] = df["thm_3"] - df["thm_5"]
    df["tmp_ud_grad"] = df[["thm_1","thm_2"]].mean(axis = 1) - df["thm_4"]

    # --- cross-modal features, temp x imu ---
    df['movement_heat_coupling'] = df["acc_mag"] * df['tmp_range']
    df['lr_movement_temp_sync'] = df["acc_mag"] / (df['tmp_lr_grad'] + 1e-8)
    df['ud_movement_temp_sync'] = df["acc_mag"] / (df['tmp_ud_grad'] + 1e-8)

    return df

def extract_tof_tensor_data(data_df, demo_df, mode = "train"):
    tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(64)]
    temp_columns = [f'thm_{i}' for i in range(1, 6)]
    tof_temp_columns = tof_columns + temp_columns
    # extract tof_data
    if mode == "train":
        sensor_data_all = data_df.dropna(subset=tof_temp_columns, how='all')
    else:
        sensor_data_all = data_df.copy()
    # merge with demographics
    sensor_merged = pd.merge(sensor_data_all, demo_df, on='subject', how='left')

    # filter for phase = gesture
    sensor_merged_sorted = sensor_merged.sort_values(['sequence_id', 'sequence_counter'])
    if mode == 'train':
      # use phase == gesture for training
      sensor_merged_sorted["is_gesture"] = (sensor_merged_sorted["phase"] == "Gesture").astype(int)
      gesture_only = sensor_merged_sorted[sensor_merged_sorted["is_gesture"] == 1].copy()
    if mode == "test":
      # use predicted is_gesture for testing
      gesture_only = sensor_merged_sorted[sensor_merged_sorted["is_gesture"] == 1].copy()

    tof_data = gesture_only[tof_columns].copy()
    tof_data.fillna(0, inplace = True)
    tof_tensor_list = []
    for seq_id, group in gesture_only.groupby('sequence_id'):
        arr = tof_data.loc[group.index].values
        # Replace -1 with 0
        arr[arr == -1] = 0
        # Normalize by dividing by 254
        arr = arr / 254.0
        # Reshape directly to (timesteps, 5 sensors, 8, 8)
        reshaped = arr.reshape(-1, 5, 8, 8)
        # Optionally move sensor axis to the end → (timesteps, 8, 8, 5)
        reshaped = np.moveaxis(reshaped, 1, -1)
        tof_tensor_list.append(tf.convert_to_tensor(reshaped))

    gesture_only.drop(columns = tof_columns, axis =1, inplace = True)
    return gesture_only, tof_tensor_list


def extract_gesture_phases_full(imu_temp_gesture_df, mode = 'train', imu_temp_sensor_means = None):
    temp_columns = [f'thm_{i}' for i in range(1, 6)]
    imu_columns = ["acc_x", "acc_y", "acc_z", "rot_w", "rot_x", "rot_y", "rot_z"]
    imu_temp_columns = temp_columns + imu_columns
    gesture_only = imu_temp_gesture_df.copy()
    # fill any NAs with sequence_id-level means, if not then global means
    # Fill with sequence-level mean
    seq_level_col_means = gesture_only.groupby('sequence_id')[imu_temp_columns].transform('mean')
    gesture_cols_filled = gesture_only[imu_temp_columns].fillna(seq_level_col_means)

    # Fill any remaining with global mean if specified/not specified
    if imu_temp_sensor_means is None:
      imu_temp_global_col_means = gesture_only[imu_temp_columns].mean()
      gesture_cols_filled = gesture_cols_filled.fillna(imu_temp_global_col_means)
    else:
      gesture_cols_filled = gesture_cols_filled.fillna(imu_temp_sensor_means)

    non_sensor_cols = gesture_only.columns.difference(imu_temp_columns)
    gesture_data = pd.concat([gesture_only[non_sensor_cols], gesture_cols_filled], axis=1)

    # add other features
    gesture_data = add_general_tabular_features(gesture_data)

    cols_to_agg = ["roll", "pitch", 'yaw',
                   'rot_magnitude', 'rot_x_norm', 'rot_y_norm', 'rot_z_norm', 'rot_w_norm',
                   'acc_mag', "acc_xy_angle", "acc_balance_score", 'acc_z_ratio',
                   "acc_x","acc_y", "acc_z",
                   'tmp_mean', 'tmp_std', 'tmp_min', 'tmp_max', 'tmp_range',
                   'tmp_front_mean', 'tmp_back_mean', 'tmp_front_back_diff', 'tmp_lr_grad', 'tmp_ud_grad',
                   'movement_heat_coupling', 'lr_movement_temp_sync', 'ud_movement_temp_sync']

    agg_stats = (
    gesture_data
    .groupby('sequence_id')[cols_to_agg]
    .agg(['mean', 'min', 'max', 'std'])
    )
    agg_stats.columns = [f"{col}_{stat}" for col, stat in agg_stats.columns]
    # add other sequence-level agg stats
    agg_stats = add_summary_imu_features(agg_stats)

    # merge back to timestep level
    gesture_data = gesture_data.merge(agg_stats, on='sequence_id', how='left')

    gesture_data.drop(columns=['row_id', 'sequence_type', 'subject', 'orientation', 'behavior', 'phase', 'is_gesture'], inplace=True, errors='ignore')

    if imu_temp_sensor_means is None:
      return gesture_data, imu_temp_global_col_means
    else:
      return gesture_data, imu_temp_sensor_means

In [ ]:
imu_temp_gesture_df, tof_gesture_tensor = extract_tof_tensor_data(train_data_df, train_demo_df, mode = "train")
imu_temp_train_data, global_train_col_means = extract_gesture_phases_full(imu_temp_gesture_df, mode = 'train')

## padding sequences for full model, tof tensor

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler


def pad_and_reshape_full(imu_temp_gesture_data, tof_gesture_tensor_list, pad_len=80, mode="train", scaler = None):
    df = imu_temp_gesture_data.copy()
    df = df.sort_values(['sequence_id', 'sequence_counter'])

    features = [col for col in df.columns if col not in ['gesture', 'gesture_label', 'sequence_id', 'sequence_counter']]
    if scaler is None:
        scaler = StandardScaler()
        df[features] = scaler.fit_transform(df[features])
    else:
        df[features] = scaler.transform(df[features])

    # pad tensors to fixed length
    X_tof_padded_list = []
    for tensor in tof_gesture_tensor_list:
        t = tensor.shape[0]
        if t < pad_len:
            padding = tf.zeros((pad_len - t, 8, 8, 5), dtype=tensor.dtype)
            padded = tf.concat([tensor, padding], axis=0)
        else:
            padded = tensor[-pad_len:]
        X_tof_padded_list.append(padded)
    X_tof_padded_arr = np.array(X_tof_padded_list)

    if mode == 'train':
        # Encode labels
        labelencoder = LabelEncoder()
        df = df.copy()
        df['gesture_label'] = labelencoder.fit_transform(df['gesture'])
        num_classes = len(labelencoder.classes_)
        # Extract and pad sequences
        X_gesture_seqs = [seq_df[features].values for _, seq_df in df.groupby('sequence_id')]
        y_gesture_seqs = [seq_df['gesture_label'].values for _, seq_df in df.groupby('sequence_id')]

        X_padded = pad_sequences(X_gesture_seqs, padding='post', dtype='float32', maxlen=pad_len, value=0.0)
        y_padded = pad_sequences(y_gesture_seqs, padding='post', dtype='int32', maxlen=pad_len, value=num_classes)

        return X_padded, X_tof_padded_arr, y_padded, labelencoder, scaler

    elif mode == 'test':
        X_gesture_seqs = [seq_df[features].values for _, seq_df in df.groupby('sequence_id')]
        X_padded = pad_sequences(X_gesture_seqs, padding='post', dtype='float32', maxlen=pad_len, value=0.0)

        return X_padded, X_tof_padded_arr


In [ ]:
_, _, _, full_labelencoder, full_scaler = pad_and_reshape_full(imu_temp_train_data, tof_gesture_tensor, pad_len = 80, mode = 'train')
del imu_temp_gesture_df, tof_gesture_tensor, imu_temp_train_data

#### load model

In [ ]:
from tensorflow.keras.models import load_model
pred_gesture_full_model = load_model('./pred_gesture_full_lstm_cnn_model.keras', custom_objects={
        "masked_loss": masked_loss,
        "masked_accuracy": masked_accuracy,
        "StopTrainingOnAccuracy": StopTrainingOnAccuracy
    })

# CMI INFERENCE

In [ ]:
import os

import pandas as pd
import polars as pl

import kaggle_evaluation.cmi_inference_server

In [ ]:
def predict(sequence: pl.DataFrame, demographics: pl.DataFrame) -> str:
    # Replace this function with your inference code.
    # You can return either a Pandas or Polars dataframe, though Polars is recommended.
    # Each prediction (except the very first) must be returned within 30 minutes of the batch features being provided.
    # extract test data
    test_data_df = sequence.to_pandas()
    test_demo_df = demographics.to_pandas()
    X_imu_phase_test = imu_sequence_pipeline(max_seq_len=700, data_df=test_data_df, demo_df=test_demo_df, mode='test', train_means=imu_phase_train_means)
    # predict gesture phase with model
    test_y_imu_phase_pred = pred_phase_imu_model.predict(X_imu_phase_test)
    # add gesture phase to original sequence df
    test_data_phase_df = reshape_gesture_pred_per_seq(test_data_df, test_y_imu_phase_pred)

    # choose model based off of data availability
    tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(64)]
    temp_columns = [f'thm_{i}' for i in range(1, 6)]
    tof_temp_columns = tof_columns + temp_columns

    # if all tof and temp data are NA, go through with IMU only model
    if test_data_phase_df[tof_temp_columns].isna().all().all():
      X_imu_gesture_test_data, _ = extract_gesture_phases(test_data_phase_df, test_demo_df, mode = 'test', imu_means = global_imu_train_col_means)
      X_imu_gesture_test_padded = pad_and_reshape(X_imu_gesture_test_data, pad_len=50, mode="test", scaler = imu_scaler)
      y_imu_gesture_preds_prob = pred_gesture_imu_model.predict(X_imu_gesture_test_padded)
      y_pred_class = sequence_mode_prediction(y_imu_gesture_preds_prob)
      return imu_labelencoder.inverse_transform(y_pred_class)
    else:
      # use full model for predictions
      X_imu_temp_gesture_test_data, X_tof_gesture_tensor = extract_tof_tensor_data(test_data_phase_df, test_demo_df, mode = "test")
      X_imu_temp_gesture_test_data_prep, _ = extract_gesture_phases_full(X_imu_temp_gesture_test_data, mode = 'test', imu_temp_sensor_means = global_train_col_means)
      X_imu_temp_gesture_full_padded, X_tof_gesture_full_padded_arr = pad_and_reshape_full(X_imu_temp_gesture_test_data_prep, X_tof_gesture_tensor, pad_len = 80, mode = 'test', scaler = full_scaler)
      y_imu_temp_gesture_preds_prob = pred_gesture_full_model.predict([X_imu_temp_gesture_full_padded, X_tof_gesture_full_padded_arr])
      y_pred_class = sequence_mode_prediction(y_imu_temp_gesture_preds_prob)
      return full_labelencoder.inverse_transform(y_pred_class)

In [ ]:
inference_server = kaggle_evaluation.cmi_inference_server.CMIInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        data_paths=(
            '/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv',
            '/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv',
        )
    )

GatewayRuntimeError: (<GatewayRuntimeErrorType.GATEWAY_RAISED_EXCEPTION: 5>, 'Traceback (most recent call last):\n  File "/content/kaggle_evaluation/core/templates.py", line 84, in run\n    predictions, row_ids = self.get_all_predictions()\n                           ^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/content/kaggle_evaluation/core/templates.py", line 55, in get_all_predictions\n    for data_batch, row_ids in self.generate_data_batches():\n  File "/content/kaggle_evaluation/cmi_gateway.py", line 51, in generate_data_batches\n    test = pl.read_csv(self.test_path)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/usr/local/lib/python3.11/dist-packages/polars/_utils/deprecation.py", line 92, in wrapper\n    return function(*args, **kwargs)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/usr/local/lib/python3.11/dist-packages/polars/_utils/deprecation.py", line 92, in wrapper\n    return function(*args, **kwargs)\n           ^^^^^^^^^^^^^^^^^^^^^^^^^\n  File "/usr/local/lib/python3.11/d